# Quantization Aware Training + Knowledge Distillation Benchmarking

In [1]:
import torch
from torch.ao.quantization.quantize_fx import convert_fx

from src.utils import load_data
from src.Quantization.utils.model_setup import setup_qat_student_model, quantization_mode
from src.utils import benchmark
from src.utils.model_setup import setup_model


 You need to install pymongo>=3.9.0 in order to use MongoOutput 


### Load Original and Quantized model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_weights = f"models/SkinCancer/Quantized/quantized_student_state.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_qat_student_model(model_name="mobilenet_v2",num_classes=num_classes)

example_inputs = next(iter(dataloaders["train"]))[0].to(device)
student_model = quantization_mode(model, "fx", example_inputs=example_inputs)

# Move the model to CPU if needed (conversion is typically done on CPU).
student_model = student_model.to("cpu")

quantized_model = convert_fx(student_model)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
quantized_model.load_state_dict(state_dict)

# Set to eval mode.
quantized_model.eval()

teacher_model = setup_model("mobilenet_v2", None, num_classes)

Model prepared using FX Graph Mode QAT.


/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/ao/quantization/utils.py:407: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/_utils.py:392: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


### Perform Benchmarking (model_size, inference time, throughput, memory usage)

In [3]:
device = torch.device("cpu")

benchmark(model1=teacher_model, model2=quantized_model, dataloader=dataloaders["test"], device=device)

Label : inference
Begin : Tue Mar 25 20:24:34 2025
Duration : 3786627.8590 us
-------------------------------
PKG :
	socket 0 :  447003876.0000 uJ


TypeError: unsupported operand type(s) for /: 'list' and 'float'